# Python API Example - End of Day - Contracts
## Calling the latest & historical End-of-Day forward curves from the Intraday Contracts Endpoint

Here we call price release data from the Spark Intraday End of Day Python API. 

__N.B. This guide is just for End-of-Day Contracts data. If you're looking for other API data products (such as Freight routes or Netbacks), please refer to their according code samples.__ 


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.

The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# Data Calling function

Here we define the function used to call the 'intraday/contracts/eod/' endpoint. This endpoint retrieves End-of-Day prices from the Intraday Contracts endpoint, and has 5 parameters - details on these parameters can be found on the API docs website:

https://www.sparkcommodities.com/api/intraday/eod.html

Users can also choose which format to retrieve the data in, 'json' or 'csv'.

__N.B__ Metadata is only available via the JSON format.


In [ ]:
## Defining the function


def fetch_historical_releases(access_token, contract, unit, start, end, format='json'):
    
    # adding all the query parameters required for the API call
    query_params = "?contract={}".format(contract)
    query_params += "&unit={}".format(unit)
    
    query_params += "&start={}".format(start)
    query_params += "&end={}".format(end)
    
    uri="/v1.0/intraday/contracts/eod/{}".format(query_params)
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = pd.DataFrame()
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data

In [ ]:
# calling data in JSON format
json_data = fetch_historical_releases(access_token, contract='jkm-ttf', unit='usd-per-mmbtu', start='2026-01-05', end='2026-01-10', format='json')
json_data

## Data included

The JSON includes 4 fields:
- 'errors': will be empty if the API call is successful, otherwise returns relevant error code
- 'data': contains the full forward curve of the requested contract
- 'metaData': provides details on the fetched data and time of API call

In [ ]:
# summary of what the JSON includes
json_data.keys()

CSV Format

Here we call historical data, and use the CSV format option to easily convert into a Pandas DataFrame

In [ ]:
# calling historical dataset, this time in CSV format
hist_df = fetch_historical_releases(access_token, contract='jkm-ttf', unit='usd-per-mmbtu', start='2025-09-20', end='2025-09-30', format='csv')
hist_df.head(5)

## N.B. Historical Data Limits

Currently, a maximum of 1 year's worth of historical data can be called at one time due to the size of the data file. 

If more data points are required, the below code can be used: the function calls the historical data 1 year at a time and combines the data into one Pandas DataFrame

In [ ]:
def loop_historical_data(access_token, contract, unit, hist_start, hist_end):
    
    # calculating the amount of days to retrieve data for
    hist_diff = (datetime.datetime.strptime(hist_end, '%Y-%m-%d') - datetime.datetime.strptime(hist_start, '%Y-%m-%d')).days
    t = 0

    starts = []
    ends = []
    
    # iterating through the historical data, 1 year at a time
    while t < hist_diff:
        # initialising dataframe
        if t == 0 and hist_diff >= 366:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=365), '%Y-%m-%d')
            hist_df = fetch_historical_releases(access_token, contract=contract, unit=unit, start=hist_start, end=diff_end, format='csv')

            starts.append(hist_start)
            ends.append(diff_end)
        
        # in case the date range is smaller than 1 year
        elif t==0 and hist_diff < 366:
            hist_df = fetch_historical_releases(access_token, contract=contract, unit=unit, start=hist_start, end=hist_end, format='csv')

        # looping through the rest of the dates
        else:
            if t < hist_diff-365:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=365), '%Y-%m-%d')
                hist2 = fetch_historical_releases(access_token, contract=contract, unit=unit, start=diff_start, end=diff_end, format='csv')
                hist_df = pd.concat([hist_df,hist2])
                starts.append(hist_start)
                ends.append(diff_end)
            # final loop
            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')
                hist2 = fetch_historical_releases(access_token, contract=contract, unit=unit, start=diff_start, end=diff_end, format='csv')
                hist_df = pd.concat([hist_df,hist2])
                starts.append(hist_start)
                ends.append(diff_end)
        
        t += 365

    hist_df['Value'] = pd.to_numeric(hist_df['Value'])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])
    hist_df = hist_df.drop_duplicates()

    return hist_df